In [50]:
import os
from datetime import datetime
from typing import Literal

from dotenv import load_dotenv
from google.genai import Client
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

True

In [10]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-3.5-flash-lite"

### Task 1

In [ ]:
# demo function replace this message with pydantic model
class TieredConversationManager:
    def __init__(self, token_budget: int):
        self.token_budget = token_budget
        self.messages = []
        self.pinned_messages = []

    def add_message(self, message, pinned=False):
        self.messages.append(message)

        if pinned:
            self.pinned_messages.append(message)

    def get_context(self):
        # Pinned messages are always preserved
        context = list(self.pinned_messages)
        tokens = sum(self.count_tokens(m) for m in context)

        # Add latest messages until budget is reached
        for message in reversed(self.messages):
            if message in self.pinned_messages:
                continue

            message_tokens = self.count_tokens(message)

            if tokens + message_tokens > self.token_budget:
                break

            context.insert(len(self.pinned_messages), message)
            tokens += message_tokens

        return context

    def count_tokens(self, message):
        # Replace with tiktoken/model tokenizer in production
        return len(message["content"].split())

In [4]:
manager = TieredConversationManager(token_budget=20)

manager.add_message({"role": "system", "content": "You are a helpful AI assistant."}, pinned=True)

for i in range(10):
    manager.add_message(
        {
            "role": "user",
            "content": f"This is conversation message number {i + 1} with some extra text.",
        }
    )

context = manager.get_context()

print("Context:")
for msg in context:
    print(msg)

print("Token count:", sum(manager.count_tokens(m) for m in context))

Context:
{'role': 'system', 'content': 'You are a helpful AI assistant.'}
{'role': 'user', 'content': 'This is conversation message number 10 with some extra text.'}
Token count: 16


### Task 2

In [44]:
class MemoryCandidate(BaseModel):
    content: str = Field(..., description="The content of the memory candidate.")
    category: Literal["preference", "fact", "event", "goal", "other"] = Field(
        ..., description="The category of the memory candidate."
    )
    confidence: float = Field(..., description="The confidence score of the memory candidate.")


class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


def build_prompt(messages: list[Message]) -> str:
    prompt = (
        "You are a memory extraction assistant. "
        "You will be given a list of conversation messages. "
        "Your task is to extract potential memory candidates from the conversation. "
        "A memory candidate is a piece of information that could be useful for future interactions. "
        "For each memory candidate, provide the content, category (preference, fact, event, goal, other), and confidence score (0-1). "
        "Return the results in given format."
    )
    prompt += "\n\nHere are the conversation messages:\n"
    for msg in messages:
        prompt += f"{msg.role}: {msg.content}\n"
    return prompt


async def extract_memory(conversations: list[Message]) -> list[MemoryCandidate]:
    prompt = build_prompt(conversations)
    response = await google_client.aio.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json", "response_schema": list[MemoryCandidate]},
    )
    return response.parsed

In [45]:
conversations = [
    # 1. User preference
    [
        Message(role="user", content="I prefer Python over JavaScript for backend development."),
        Message(role="assistant", content="Python is a great choice for backend development."),
        Message(
            role="user",
            content="I'm currently learning FastAPI and want to use it for my AI projects.",
        ),
        Message(role="assistant", content="FastAPI works well for building AI APIs."),
        Message(role="user", content="Please use Python examples whenever possible."),
        Message(role="assistant", content="Sure, I'll use Python examples when appropriate."),
    ],
    # 2. Personal / stable information
    [
        Message(role="user", content="I'm currently studying BCA."),
        Message(role="assistant", content="That's great! What area are you focusing on?"),
        Message(role="user", content="I'm mainly interested in AI and machine learning."),
        Message(role="assistant", content="AI and ML are excellent areas to explore."),
        Message(role="user", content="My goal is to become an AI Engineer."),
        Message(
            role="assistant",
            content="Then building strong Python, ML, and LLM skills will be useful.",
        ),
    ],
    # 3. Project-related memory
    [
        Message(role="user", content="I'm building a RAG chatbot for company documents."),
        Message(role="assistant", content="A hybrid retrieval pipeline could work well."),
        Message(role="user", content="I'm using PostgreSQL with PGVector for vector storage."),
        Message(role="assistant", content="PGVector is a good option for this architecture."),
        Message(role="user", content="I also want to add keyword search and reranking."),
        Message(
            role="assistant", content="That would give you a strong hybrid retrieval pipeline."
        ),
    ],
    # 4. Temporary information
    [
        Message(role="user", content="I have an interview tomorrow."),
        Message(role="assistant", content="Good luck! What role is it for?"),
        Message(role="user", content="It's for a Python developer position."),
        Message(role="assistant", content="You should revise Python fundamentals and APIs."),
        Message(role="user", content="After the interview I'll continue working on my project."),
        Message(role="assistant", content="Sounds good. Focus on the interview first."),
    ],
    # 5. Mixed conversation
    [
        Message(role="user", content="I like concise answers when I'm debugging code."),
        Message(role="assistant", content="I'll keep debugging answers concise."),
        Message(role="user", content="My current project uses FastAPI, PostgreSQL, and Redis."),
        Message(role="assistant", content="That's a common production-oriented stack."),
        Message(
            role="user", content="Yesterday I spent three hours fixing a Redis connection issue."
        ),
        Message(role="assistant", content="Connection issues can definitely take time to debug."),
    ],
]

In [48]:
MemoryCandidateList = []

for conversation in conversations:
    try:
        memory_candidates = await extract_memory(conversation)
        MemoryCandidateList.extend(memory_candidates)
    except ValidationError as e:
        print(f"Validation error: {e}")

In [49]:
MemoryCandidateList

[MemoryCandidate(content='Prefers Python over JavaScript for backend development', category='preference', confidence=0.99),
 MemoryCandidate(content='Currently learning FastAPI', category='fact', confidence=0.95),
 MemoryCandidate(content='Wants to use FastAPI for AI projects', category='goal', confidence=0.95),
 MemoryCandidate(content='Prefers Python examples whenever possible', category='preference', confidence=0.99),
 MemoryCandidate(content='The user is studying BCA.', category='fact', confidence=1.0),
 MemoryCandidate(content='The user is interested in AI and machine learning.', category='preference', confidence=1.0),
 MemoryCandidate(content="The user's goal is to become an AI Engineer.", category='goal', confidence=1.0),
 MemoryCandidate(content='The user is building a RAG chatbot for company documents.', category='goal', confidence=0.99),
 MemoryCandidate(content='The user is using PostgreSQL with PGVector for vector storage.', category='fact', confidence=0.99),
 MemoryCandida

### Task 3

In [ ]:
schema = """
CREATE TABLE user_memories (
 id BIGSERIAL PRIMARY KEY,
 user_id UUID NOT NULL,
 content TEXT NOT NULL,
 category TEXT NOT NULL,
 embedding VECTOR(1536) NOT NULL,
 confidence FLOAT NOT NULL,
 created_at TIMESTAMPTZ DEFAULT now(),
 last_accessed_at TIMESTAMPTZ DEFAULT now(),
 access_count INT DEFAULT 0
);

CREATE INDEX ON user_memories (user_id);
CREATE INDEX ON user_memories USING hnsw (embedding vector_cosine_ops);
"""

In [ ]:
def store_memory(
    db,
    user_id,
    content: str,
    category: str,
    embedding: list[float],
    confidence: float,
):
    # replace the logic to store the memory in the database
    memory = {
        "user_id": user_id,
        "content": content,
        "category": category,
        "embedding": embedding,
        "confidence": confidence,
        "created_at": datetime.utcnow(),  # noqa: DTZ003
        "last_accessed_at": datetime.utcnow(),  # noqa: DTZ003
        "access_count": 0,
    }

    return memory


def retrieve_memories(
    db,
    user_id,
    query_embedding: list[float],
    limit: int = 5,
):
    # replace the logic to retrieve memories from the database
    memories = [
        {
            "user_id": user_id,
            "content": "I prefer Python over JavaScript for backend development.",
            "category": "preference",
            "embedding": [0.1, 0.2, 0.3],
            "confidence": 0.9,
        }
    ]

    return memories

### Task 4

implemented in test_exercise.py

### Task 5

In [55]:
class Episode(BaseModel):
    task: str
    action: str
    outcome: str


class EpisodicMemory:
    def __init__(self):
        self.episodes: list[Episode] = []

    def store(
        self,
        task: str,
        action: str,
        outcome: str,
    ):
        self.episodes.append(
            Episode(
                task=task,
                action=action,
                outcome=outcome,
            )
        )

    def has_already_tried(
        self,
        task: str,
        action: str,
    ) -> bool:
        return any(episode.task == task and episode.action == action for episode in self.episodes)

    def get_history(self, task: str):
        return [episode for episode in self.episodes if episode.task == task]

### Task 6

implemented in test_exercise_6.py

### Task 7

In [57]:
class UserMemory(BaseModel):
    id: int
    user_id: str
    content: str
    category: str
    confidence: float


def what_do_you_remember(
    user_id: str,
) -> list[dict]:
    """
    Return all long-term memories stored for a user.
    """

    memories: list[UserMemory] = []  # replace this with actual retrieval logic from the database
    user_memories = [memory for memory in memories if memory.user_id == user_id]

    return [
        {
            "id": memory.id,
            "memory": memory.content,
            "category": memory.category,
        }
        for memory in user_memories
    ]